### Building Core Metrics

This will focus on building the calculated fields needed for Power BI visualization, lessening the need for DAX multi-step calculations

The following calculations center around the project's success criteria questions to be answered:

    1. Which 3 retail categories had the lowest YoY turnover growth nationally in the most recent 12 months?
    2. Which category has the largest gap vs the Total Industry benchmark, and by how many percentage
    points?
    3. What is the 12-month rolling trend for Food Retailing vs Clothing, Footwear & Personal Accessories?
    4. Which month does each category peak in, and what does that mean for promotional timing?
    5. What is the CFO recommendation for promotional budget reallocation, backed by 3 specific data
    points?


### Import libraries and create path to processed data

In [2]:
import pandas as pd
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

### Rolling 12-Month YoY Growth

💡 What is each category's rolling YoY Grwoth?

This shows each category's growth performance YoY as a percentage calculated.

A 12-month window was chosen to minimize the impact of seasonality on an otherwise monthly analysis. It would be difficult to determine whether growth comes from seasonal demand or an unnrelated market growth.

In [16]:
# Read processed CSV into dataframe
df_all = pd.read_csv(DATA_PROCESSED / 'retail_clean.csv', parse_dates=['date'])
print(df_all)
df_all = df_all.sort_values(['category', 'date'])

# Save Total (Industry) data 
df_total = df_all[df_all['category'] == 'Total (Industry)'].copy()

# Remove Total (Industry) from the main dataframe
df = df_all[df_all['category'] != 'Total (Industry)'].copy()


# For each category, sum the last 12 months of turnover into a single rolling figure
# min_periods=12 means don't calculate until we have a full 12 months of data
df['r12m_turnover'] = (
    df.groupby('category')['turnover_m']
    .transform(lambda x: x.rolling(12, min_periods=12).sum())
)

# .shift(12) moves each value down 12 rows — giving us the rolling total from a year ago
df['r12m_prior'] = (
    df.groupby('category')['r12m_turnover']
    .shift(12)
)

# YoY growth = (this year - last year) / last year × 100
df['yoy_growth_pct'] = (
    (df['r12m_turnover'] - df['r12m_prior']) / df['r12m_prior'] * 100
    ).round(1)

df.to_csv(DATA_PROCESSED / 'retail_yoy_growth.csv', index=False)
print('Saved: retail_yoy_growth.csv')

           date  series_id  turnover_m  \
0    1982-04-01  A3348636C       342.4   
1    1982-05-01  A3348636C       342.1   
2    1982-06-01  A3348636C       328.7   
3    1982-07-01  A3348636C       338.5   
4    1982-08-01  A3348636C       331.5   
...         ...        ...         ...   
3628 2025-02-01  A3348582J     32744.3   
3629 2025-03-01  A3348582J     36277.1   
3630 2025-04-01  A3348582J     35382.3   
3631 2025-05-01  A3348582J     36907.1   
3632 2025-06-01  A3348582J     36351.9   

                                           category series_type  year  month  \
0     Cafes, restaurants and takeaway food services    Original  1982      4   
1     Cafes, restaurants and takeaway food services    Original  1982      5   
2     Cafes, restaurants and takeaway food services    Original  1982      6   
3     Cafes, restaurants and takeaway food services    Original  1982      7   
4     Cafes, restaurants and takeaway food services    Original  1982      8   
...            

In [6]:
# Read csv data into dataframe, remove Total (Industry)
df = pd.read_csv(DATA_PROCESSED / 'retail_clean.csv', parse_dates=['date'])
df = df[df['category'] != 'Total (Industry)'].copy()


# Sort so dates are in order within each category for rolling() function
df = df.sort_values(['category', 'date'])

# For each category, sum the last 12 months of turnover into a single rolling figure
# min_periods=12 means don't calculate until we have a full 12 months of data
df['r12m_turnover'] = (
    df.groupby('category')['turnover_m']
    .transform(lambda x: x.rolling(12, min_periods=12).sum())
)

# .shift(12) moves each value down 12 rows — giving us the rolling total from a year ago
df['r12m_prior'] = (
    df.groupby('category')['r12m_turnover']
    .shift(12)
)

# YoY growth = (this year - last year) / last year × 100
df['yoy_growth_pct'] = (
    (df['r12m_turnover'] - df['r12m_prior']) / df['r12m_prior'] * 100
    ).round(1)

# Save output to CSV
df.to_csv(DATA_PROCESSED / 'retail_yoy_growth.csv', index=False)
print('Saved: retail_yoy_growth.csv')

Saved: retail_yoy_growth.csv


### Cateogry Share of Total Retail

💡 What percent of total retail does each category represent?

This shows us each category's share or grasp over the total retail industry. This indicates which categories have the most profit and/or market dominance


In [5]:
# Find total turnover per category, not including the Total (Industry)
total = (
    df[df['category'] == 'Total (Industry)']
    .groupby('date')['turnover_m']
    .sum()
    .reset_index()
    .rename(columns={'turnover_m': 'total_m'})
)

# Create dataframe of retail categories
df_cats = df[df['category'] != 'Total (Industry)'].copy()
df_cats = df_cats.merge(total, on='date', how='left')

# Create datamframe of each category's retail share as percentage (category turnover / overall turnover) * 100
df_cats['category_share_pct'] = (
    df_cats['turnover_m'] / df_cats['total_m'] * 100
).round(1)

# Save output as csv
df_cats.to_csv(DATA_PROCESSED / 'retail_category_share.csv', index=False)
print('Saved: retail_category_share.csv')

Saved: retail_category_share.csv


### Category Underperforming Ranking

💡 How far above or below does each category's YoY growth sit in comparison to total industry growth?

This puts each category's YoY growth in the perspective of the total industry's growth (as calculated using the Total (Industry) data)

In [ ]:
# Calculate rolling YoY growth for Total (Industry) as the national benchmark
df_total = df_total.sort_values('date')

df_total['r12m_turnover'] = df_total['turnover_m'].rolling(12, min_periods=12).sum()
df_total['r12m_prior'] = df_total['r12m_turnover'].shift(12)

df_total['yoy_growth_pct'] = ((df_total['r12m_turnover'] - df_total['r12m_prior']) / df_total['r12m_prior'] * 100
    ).round(1)

# Calculate the total industry's YoY growth to serve as the benchmark
benchmark = df_total[df_total['date'] == df_total['date'].max()]['yoy_growth_pct'].values[0]

# Compare each category's latest YoY growth to the benchmark
latest = df[df['date'] == df['date'].max()].copy()
latest['benchmark_growth_pct'] = benchmark

# gap = how many ppt above or below the national benchmark each category sits
# negative gap = underperforming — these are the CFO brief categories

latest['gap_vs_national_ppt'] = (latest['yoy_growth_pct'] - benchmark).round(1)
latest = latest.sort_values('yoy_growth_pct') # worst performers at top

# Save output to CSV
latest.to_csv(DATA_PROCESSED / 'retail_category_ranking.csv', index=False)
print('Saved: retail_category_ranking.csv')

           date  series_id  turnover_m          category series_type  year  \
3114 1982-04-01  A3348582J      3396.4  Total (Industry)    Original  1982   
3115 1982-05-01  A3348582J      3497.9  Total (Industry)    Original  1982   
3116 1982-06-01  A3348582J      3357.8  Total (Industry)    Original  1982   
3117 1982-07-01  A3348582J      3486.8  Total (Industry)    Original  1982   
3118 1982-08-01  A3348582J      3355.9  Total (Industry)    Original  1982   
...         ...        ...         ...               ...         ...   ...   
3628 2025-02-01  A3348582J     32744.3  Total (Industry)    Original  2025   
3629 2025-03-01  A3348582J     36277.1  Total (Industry)    Original  2025   
3630 2025-04-01  A3348582J     35382.3  Total (Industry)    Original  2025   
3631 2025-05-01  A3348582J     36907.1  Total (Industry)    Original  2025   
3632 2025-06-01  A3348582J     36351.9  Total (Industry)    Original  2025   

      month month_name  r12m_turnover  r12m_prior  yoy_growth_p